# **FEW-SHOT CLASSIFICATION FOR LLAMA-3.2-3B**


In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('fpb_test.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...


### libraries

'transformers' - for model

'accelerate' - optimise model loading

In [3]:
# install transformers
!pip install transformers accelerate -q

### huggingface login

run this in colab with HF_TOKEN as a secret

In [4]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab's secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")

Successfully logged in to Hugging Face.


### load llama model

In [5]:
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import logging
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity(logging.ERROR)

# configure model
model_id = "meta-llama/Llama-3.2-3B-Instruct" # it has to be the instruct model

# load model and tokeniser
tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"

# load in 4-bit to standard colab gpu memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)

# Initialize the text generation pipeline once globally
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
    torch_dtype=torch.bfloat16,
    pad_token_id=tokenizer.eos_token_id
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.8 MB/s eta 0:00:00


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

### data prep, few-shot

create a prompt that includes a few examples of the task (the 'shots') and then the text we want the model to classify

**IMPORTANT: EACH EXAMPLE SHOULD BE OF A UNIQUE CATEGORY**

split the dataset into a small set for examples and the rest for classification

In [7]:
# unique labels are...
print("Unique labels in the dataset:", df['label'].unique())
unique_labels = df['label'].unique().tolist()

df_neutral = df[df['label'] == 'Neutral'].copy()
df_fear = df[df['label'] == 'Fear'].copy()
df_optimism = df[df['label'] == 'Optimism'].copy()
df_sadness = df[df['label'] == 'Sadness'].copy()
df_joy = df[df['label'] == 'Joy'].copy()

neutral_example = df_neutral.sample(n=1, random_state=30) # reproducibility
fear_example = df_fear.sample(n=1, random_state=30)
optimism_example = df_optimism.sample(n=1, random_state=30)
sadness_example = df_sadness.sample(n=1, random_state=30)
joy_example = df_joy.sample(n=1, random_state=30)

few_shot_examples = pd.concat([neutral_example, fear_example, optimism_example, sadness_example, joy_example])

# # Shuffle the dataframe to ensure random selection of few-shot examples
# df_shuffled = df.sample(frac=1, random_state=30).reset_index(drop=True)

# # Select a few examples for few-shot prompting
# num_few_shot_examples = 5 # 10 would be too much and take too long
# few_shot_examples = df_shuffled.head(num_few_shot_examples)
# inference_data = df_shuffled.tail(len(df_shuffled) - num_few_shot_examples).reset_index(drop=True)

inference_data = df.copy()

for r in few_shot_examples.itertuples(index=False):
    inference_data = inference_data[inference_data.text != r.text]

Unique labels in the dataset: ['Neutral' 'Fear' 'Optimism' 'Sadness' 'Joy']


### few-shot inference

iterate through inference_data

In [10]:
def classify(text, candidate_labels):

    # Update prompt to explicitly ask for one of the candidate labels
    # And make sure the system prompt is aligned with the desired single-word output
    prompt_text = f"Here are some examples of texts that have been classified.\n"

    for sample in few_shot_examples.itertuples():
        prompt_text += f"Text: '{sample.text}'\n"
        prompt_text += f"Category: {sample.label}\n\n"

    prompt_text = f"Now, classify the sentiment of the following text into one of these categories: {', '.join(candidate_labels)}."

    prompt = [
        {"role": "system", "content": "You are a helpful assistant that classifies text. Respond with only the category name."},
        {"role": "user", "content": f"{prompt_text}\nText: '{text}'"},
    ]

    # Use the globally initialized generator

    generation = generator(
        prompt,
        do_sample=True, # Explicitly set do_sample to True for more diverse responses
        temperature=1.0, # Further increased temperature for more diverse responses
        top_p=0.9, # Slightly reduced top_p to focus sampling on more probable tokens
        max_new_tokens=5
    )

    # Extract the generated text from the full conversation history
    full_conversation = generation[0]['generated_text']
    #print(full_conversation) - debug

    predicted_text = ""
    for message in reversed(full_conversation):
        if message.get('role') == 'assistant':
            predicted_text = message.get('content', '').strip()
            break

    #print(predicted_text) - debug

    # Attempt to find an exact match for the predicted text within the candidate labels
    predicted_label = "Unknown"
    predicted_text_lower = predicted_text.lower()
    for label in candidate_labels:
        if label.lower() == predicted_text_lower:
            predicted_label = label
            break

    return predicted_label

# time to evaluate!
y_true = inference_data['label'].tolist()
y_pred = [classify(text, unique_labels) for text in inference_data['text']]

# print(y_true)
# print(y_pred)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [11]:
# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=unique_labels, zero_division=0)

# Create a DataFrame for per-class F1 scores
per_class_f1_df = pd.DataFrame({
    'Category': unique_labels,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Support': support
})

print("Per-class F1 Scores:")
display(per_class_f1_df)

# For overall metrics, we can still calculate them (e.g., weighted average)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', labels=unique_labels, zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', labels=unique_labels, zero_division=0)

# export metrics (overall weighted metrics)
results_dict = {
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'] and ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'],
    'Value': [accuracy, precision_weighted, recall_weighted, f1_weighted, precision_macro, recall_macro, f1_macro]
}

# create dataframe
results_df = pd.DataFrame(results_dict)
display(results_df)

dashboard_dict = {
    'model': 'Llama 3.2 3B',
    'strategy': 'few-shot',
    'accuracy': accuracy,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'f1_fear': per_class_f1_df.at[1, 'F1 Score'],
    'f1_joy': per_class_f1_df.at[4, 'F1 Score'],
    'f1_neutral': per_class_f1_df.at[0, 'F1 Score'],
    'f1_optimism': per_class_f1_df.at[2, 'F1 Score'],
    'f1_sadness': per_class_f1_df.at[3, 'F1 Score']
}

dashboard_df = pd.DataFrame(dashboard_dict, index=[0])

# export to csv
dashboard_df.to_csv('few_shot_llama_dashboard.csv', index=False)

print("Results saved to 'few_shot_llama_dashboard.csv'")
display(dashboard_df)

Per-class F1 Scores:


,Category,Precision,Recall,F1 Score,Support
0,Neutral,0.614035,0.974478,0.753363,431
1,Fear,0.100000,0.333333,0.153846,6
2,Optimism,0.529412,0.046154,0.084906,195
3,Sadness,1.000000,0.012195,0.024096,82
4,Joy,0.000000,0.000000,0.000000,8


,Metric,Value
0,Accuracy,0.598338
1,Precision (Weighted),0.623940
2,Recall (Weighted),0.598338
3,F1 Score (Weighted),0.476669
4,Precision (Macro),0.448689
5,Recall (Macro),0.273232
6,F1 Score (Macro),0.203242


Results saved to 'few_shot_llama_dashboard.csv'


,model,strategy,accuracy,f1_macro,f1_weighted,f1_fear,f1_joy,f1_neutral,f1_optimism,f1_sadness
0,Llama 3.2 3B,few-shot,0.598338,0.203242,0.476669,0.153846,0.0,0.753363,0.084906,0.024096
